# 04 · RAG Pipeline on a Document

Chunking, embeddings, a Chroma vector database, a RAG chain, and RAG with memory.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AfzalManihar/Agentic-AI-Learning/blob/main/04_rag_pipeline_on_document.ipynb)

**Covered in this notebook**

- Build the RAG Pipeline on a Document

> Part of [Agentic AI Learning](https://github.com/AfzalManihar/Agentic-AI-Learning). Add your own API keys in Colab Secrets (key icon) before running.

# Build the RAG Pipeline on a Document

In [ ]:
# Step 1: Our document (could be pdf, csv, plain text)
my_text = """
Artificial Intelligence is a branch of computer science focused on building
machines that can perform tasks that typically require human intelligence.

Machine Learning is a subset of AI where systems learn from data instead of
being explicitly programmed. The more data you give it, the better it gets.

Deep Learning is a subset of Machine Learning that uses neural networks with
many layers. It is especially good at image and speech recognition.

Natural Language Processing (NLP) is the branch of AI that deals with text
and language. ChatGPT is an example of an NLP model.

Reinforcement Learning is a type of Machine Learning where an agent learns
by taking actions and receiving rewards or punishments. It is used in
robotics and games.
"""

In [ ]:
# Chunking : Breaking document into pieces or chunks
!pip install -U langchain_text_splitters -q

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter # For breaking into chunks
from langchain_community.vectorstores import Chroma # Storing the vector embeddings into Vector Database
from langchain_core.documents import Document # Storing our simple (word) document

In [ ]:
# Splitting into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=200)
chunks = splitter.split_text(my_text)

In [ ]:
for chunk in chunks:
    print(chunk)
    print('----------------------------------------')

#Our text has been divided into chunks.


Artificial Intelligence is a branch of computer science focused on building
machines that can perform tasks that typically require human intelligence.
----------------------------------------
Machine Learning is a subset of AI where systems learn from data instead of
being explicitly programmed. The more data you give it, the better it gets.
----------------------------------------
Deep Learning is a subset of Machine Learning that uses neural networks with
many layers. It is especially good at image and speech recognition.
----------------------------------------
Natural Language Processing (NLP) is the branch of AI that deals with text
and language. ChatGPT is an example of an NLP model.
----------------------------------------
Reinforcement Learning is a type of Machine Learning where an agent learns
by taking actions and receiving rewards or punishments. It is used in
robotics and games.
----------------------------------------


In [ ]:
# Vector Database
!pip install chromadb -q

ChromaDB khud ek Vector Database hai.

Flow:

Text → Chunks → Documents → Embeddings → ChromaDB (Vector DB)

In [ ]:
# Step 3: Store the chunks
docs = [Document(page_content=chunk) for chunk in chunks]

vectorStore = Chroma.from_documents(docs, embedding=embedder)

In [ ]:
retriever = vectorStore.as_retriever(search_kwargs={'k':2})
# Retrieve the top 2 most matching chunks according to user query

In [ ]:
question = 'What is deep learning?'

relevant_chunks = retriever.invoke(question)

relevant_chunks

[Document(metadata={}, page_content='Deep Learning is a subset of Machine Learning that uses neural networks with\nmany layers. It is especially good at image and speech recognition.'),
 Document(metadata={}, page_content='Machine Learning is a subset of AI where systems learn from data instead of\nbeing explicitly programmed. The more data you give it, the better it gets.')]

In [ ]:
pip install langchain-groq -q

In [ ]:
from google.colab import userdata
my_api_key = userdata.get('groq_api')

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = 'openai/gpt-oss-20b',
    temperature = 0, # Creativity level of the model
    api_key = my_api_key
)

In [ ]:
# Retrieval + Augmentation + Generation
def answer_from_docs(question: str) -> str:
    relevant_chunks = retriever.invoke(question)

    prompt = f'''Answer the question using only the context below.


                Context:
                {relevant_chunks}

                Question:
                {question}
            '''

    response = llm.invoke(prompt)
    return response

In [ ]:
print(answer_from_docs("What is the difference between AI and Machine Learning?").content)

Artificial Intelligence (AI) is a broad field of computer science that aims to build machines capable of performing tasks that normally require human intelligence.  
Machine Learning (ML) is a specific subset of AI: it focuses on systems that learn from data rather than being explicitly programmed. In other words, ML is one approach to achieving AI, and it improves as it receives more data.


In [ ]:
print(answer_from_docs("What is capital of France?").content)

# It is not able to answer because our text does not contain any information about “What is the capital of France?”
#Our text only contains information about AI, ML, and DL.


I’m sorry, but the provided context does not contain information about the capital of France.


In [ ]:
llm.invoke('What is capital of France?').content

# llm.invoke() uses the LLM's own knowledge, while RAG answers using the provided document.

'The capital of France is **Paris**.'

In [ ]:
# Challenge 1 (Easy)
# Add 2–3 sentences about yourself (your name, what you study, your city) as the document text.
# Build the RAG pipeline on it.
# Then ask questions like "Where does this person live?"
# and confirm it answers from your text.

In [ ]:
my_text = """
My name is Afzal Manihar. I am studying Data Science with Generative AI.
I live in Mumbai, India. I am interested in Machine Learning and Generative AI.
"""

In [ ]:
# Chunking : Breaking document into pieces or chunks
!pip install -U langchain_text_splitters -q

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter # For breaking into chunks
from langchain_community.vectorstores import Chroma # Storing the vector embeddings into Vector Database
from langchain_core.documents import Document # Storing our simple (word) document

In [ ]:
# Splitting into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=200)
chunks = splitter.split_text(my_text)

In [ ]:
for chunk in chunks:
    print(chunk)
    print('----------------------------------------')


My name is Afzal Manihar. I am studying Data Science with Generative AI.
I live in Mumbai, India. I am interested in Machine Learning and Generative AI.
----------------------------------------


In [ ]:
# Vector Database
!pip install chromadb -q

In [ ]:
# Step 3: Store the chunks
docs = [Document(page_content=chunk) for chunk in chunks]

vectorStore = Chroma.from_documents(docs, embedding=embedder)

In [ ]:
retriever = vectorStore.as_retriever(search_kwargs={'k':2})
# Retrieve the top 2 most matching chunks according to user query

In [ ]:
question = "Where does this person live?"
print(answer_from_docs(question).content)

Mumbai, India.


In [ ]:
question = "What does Afzal study?"
print(answer_from_docs(question).content)

Afzal is studying **Data Science with Generative AI**.


In [ ]:
# RAG + Memory
# Challenge 2 (Super Tough):
# Create the RAG as a tool
# and add it to your langgraph
# You need to ask it 3 questions
# First question should be from doc
# Second question should be a follow up question from doc
# Third question should not be in doc, it should need web search tool

In [ ]:
# my_text = """
# Python is a high-level, interpreted programming language known for its simple syntax.
# It was created by Guido van Rossum and first released in 1991.
# Python is widely used in data science, web development, and automation.
# The latest major version is Python 3, which introduced many improvements over Python 2.
# Popular Python libraries include NumPy, Pandas, and Scikit-learn for data science.
# """

# "Who created Python?",
# "What did that person release and in what year?",  # needs memory of "Guido"
# "What is the current weather in Bangalore?"         # not in doc, needs web search

In [ ]:
my_text = """
Python is a high-level, interpreted programming language known for its simple syntax.
It was created by Guido van Rossum and first released in 1991.
Python is widely used in data science, web development, and automation.
The latest major version is Python 3, which introduced many improvements over Python 2.
Popular Python libraries include NumPy, Pandas, and Scikit-learn for data science.
"""

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Split document into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=0
)

chunks = splitter.split_text(my_text)

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

# Convert chunks into documents
docs = [Document(page_content=chunk) for chunk in chunks]

# Store documents in vector database
vectorStore = Chroma.from_documents(
    docs,
    embedding=embedder
)

retriever = vectorStore.as_retriever(
    search_kwargs={"k": 2}
)

In [ ]:
from langchain_groq import ChatGroq

# Create the base LLM
base_llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    api_key=my_api_key
)

In [ ]:
def answer_from_docs(question: str):
    # Retrieve relevant chunks
    relevant_chunks = retriever.invoke(question)

    prompt = f"""
Answer using only the context below.

Context:
{relevant_chunks}

Question:
{question}
"""

    return base_llm.invoke(prompt)

In [ ]:
from langchain_core.tools import tool

@tool
def rag_search(query: str) -> str:
    """Search the provided Python document for relevant information."""
    print("🔎 RAG Search called")
    return answer_from_docs(query).content

In [ ]:
@tool
def web_search(query: str) -> str:
    """Search the web for current or outside information."""
    print("⭐ Web Search called")

    try:
        return search.run(query)
    except Exception as e:
        return f"Web search failed: {e}"

In [ ]:
tools = [rag_search, web_search]

# Give tools to the LLM
llm = base_llm.bind_tools(tools)

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
def assistant_node(state: State):
    system_message = """
You are an AI assistant with access to two tools.

Use rag_search for information available in the Python document.
Use web_search for current information or information not available in the document.
Choose the appropriate tool yourself.
"""

    response = llm.invoke(
        [("system", system_message)] + state["messages"]
    )

    return {"messages": [response]}

In [ ]:
from langgraph.prebuilt import ToolNode

tool_node = ToolNode(tools)

In [ ]:
from langgraph.graph import END

def should_continue(state: State):
    # Continue if the LLM requested a tool
    if state["messages"][-1].tool_calls:
        return "tools"

    return END

In [ ]:
from langgraph.graph import StateGraph, START

builder = StateGraph(State)

builder.add_node("assistant", assistant_node)
builder.add_node("tools", tool_node)

builder.add_edge(START, "assistant")
builder.add_conditional_edges("assistant", should_continue)
builder.add_edge("tools", "assistant")

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

# Compile graph with memory
graph_with_memory = builder.compile(
    checkpointer=memory
)

In [ ]:
config = {
    "configurable": {
        "thread_id": "rag_challenge_2"
    }
}

In [ ]:
response = graph_with_memory.invoke(
    {"messages": [("user", "Who created Python?")]},
    config=config
)

print(response["messages"][-1].content)

⭐ Web Search called
Python was created by **Guido van Rossum**.  
He began developing the language in December 1989 while working at the Centrum Wiskunde & Informatica (CWI) in the Netherlands, and released the first version (0.9.0) in February 1991. Van Rossum has been the principal “Benevolent Dictator For Life” (BDFL) of Python until stepping down from that role in 2018, after which the language is governed by the Python Steering Council.


In [ ]:
response = graph_with_memory.invoke(
    {"messages": [("user", "What did that person release and in what year?")]},
    config=config
)

print(response["messages"][-1].content)

⭐ Web Search called
Guido van Rossum released the first public version of Python—**Python 0.9.0**—in **1991** (February 20, 1991).


In [ ]:
response = graph_with_memory.invoke(
    {"messages": [("user", "What is the current weather in Bangalore?")]},
    config=config
)

print(response["messages"][-1].content)

⭐ Web Search called
⭐ Web Search called
I’m sorry, but I’m not able to retrieve real‑time data such as the current weather.  
To get the up‑to‑date weather for Bangalore, you can check a reliable weather website (e.g., Weather.com, AccuWeather, or the official Indian Meteorological Department site) or use a weather app on your phone.
